In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
n_epoch = 100
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="pinv")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.5972690582275391
epoch:  1, loss: 0.5076414346694946
epoch:  2, loss: 0.4049513339996338
epoch:  3, loss: 0.33777955174446106
epoch:  4, loss: 0.2998155951499939
epoch:  5, loss: 0.2651575803756714
epoch:  6, loss: 0.23778152465820312
epoch:  7, loss: 0.21582536399364471
epoch:  8, loss: 0.21281009912490845
epoch:  9, loss: 0.209098219871521
epoch:  10, loss: 0.177684485912323
epoch:  11, loss: 0.153955340385437
epoch:  12, loss: 0.13230890035629272
epoch:  13, loss: 0.10965337604284286
epoch:  14, loss: 0.09275098145008087
epoch:  15, loss: 0.07693290710449219
epoch:  16, loss: 0.06319323182106018
epoch:  17, loss: 0.05403657257556915
epoch:  18, loss: 0.03867952525615692
epoch:  19, loss: 0.0337991937994957
epoch:  20, loss: 0.029316332191228867
epoch:  21, loss: 0.025194542482495308
epoch:  22, loss: 0.020864436402916908
epoch:  23, loss: 0.00560377910733223
epoch:  24, loss: 0.005366450175642967
epoch:  25, loss: 0.004419549368321896
epoch:  26, loss: 0.003354955

In [11]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9819546814723323
Test metrics:  R2 = 0.9265445146323547


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="pinv-trunc")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.09824060648679733
epoch:  1, loss: 0.040511198341846466
epoch:  2, loss: 0.02976936101913452
epoch:  3, loss: 0.01846635900437832
epoch:  4, loss: 0.014110589399933815
epoch:  5, loss: 0.00983830913901329
epoch:  6, loss: 0.008382881060242653
epoch:  7, loss: 0.007414206862449646
epoch:  8, loss: 0.006440572906285524
epoch:  9, loss: 0.005838640965521336
epoch:  10, loss: 0.005338381510227919
epoch:  11, loss: 0.004959415178745985
epoch:  12, loss: 0.004658577032387257
epoch:  13, loss: 0.004398704972118139
epoch:  14, loss: 0.0041524325497448444
epoch:  15, loss: 0.00389929860830307
epoch:  16, loss: 0.0036426803562790155
epoch:  17, loss: 0.0034136204048991203
epoch:  18, loss: 0.0032182070426642895
epoch:  19, loss: 0.003062134375795722
epoch:  20, loss: 0.0029187477193772793
epoch:  21, loss: 0.0027785166166722775
epoch:  22, loss: 0.0026520765386521816
epoch:  23, loss: 0.0025321352295577526
epoch:  24, loss: 0.0024238931946456432
epoch:  25, loss: 0.00232957163

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.993072931048352
Test metrics:  R2 = 0.9858724014147039


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="solve")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.5651689171791077
epoch:  1, loss: 0.4919569790363312
epoch:  2, loss: 0.3200645446777344
epoch:  3, loss: 0.25027287006378174
epoch:  4, loss: 0.03234129399061203
epoch:  5, loss: 0.02938910759985447
epoch:  6, loss: 0.02691861242055893
epoch:  7, loss: 0.024883586913347244
epoch:  8, loss: 0.012808860279619694
epoch:  9, loss: 0.00945853628218174
epoch:  10, loss: 0.007810345850884914
epoch:  11, loss: 0.006931556388735771
epoch:  12, loss: 0.0062732999213039875
epoch:  13, loss: 0.005720253102481365
epoch:  14, loss: 0.005230134353041649
epoch:  15, loss: 0.004841942340135574
epoch:  16, loss: 0.004535962827503681
epoch:  17, loss: 0.0042707365937530994
epoch:  18, loss: 0.004072450567036867
epoch:  19, loss: 0.003890764433890581
epoch:  20, loss: 0.0037586665712296963
epoch:  21, loss: 0.0036469257902354
epoch:  22, loss: 0.0035290869418531656
epoch:  23, loss: 0.003442745190113783
epoch:  24, loss: 0.003368535079061985
epoch:  25, loss: 0.003355132881551981
epoch

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9667404916273752
Test metrics:  R2 = 0.8564876116110248


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="cholesky")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.3114284574985504
epoch:  1, loss: 0.28853368759155273
epoch:  2, loss: 0.22984448075294495


Linear algebra error. linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 140 is not positive-definite).
Fallback to lsqrs solver.


epoch:  3, loss: 0.05502967908978462
epoch:  4, loss: 0.028439750894904137
epoch:  5, loss: 0.025183331221342087
epoch:  6, loss: 0.0102538438513875
epoch:  7, loss: 0.00994668249040842
epoch:  8, loss: 0.008105579763650894
epoch:  9, loss: 0.00787637010216713
epoch:  10, loss: 0.007665586657822132
epoch:  11, loss: 0.006266193464398384
epoch:  12, loss: 0.00561218848451972
epoch:  13, loss: 0.004972724709659815
epoch:  14, loss: 0.004603953566402197
epoch:  15, loss: 0.004359907936304808
epoch:  16, loss: 0.004020656459033489
epoch:  17, loss: 0.0038681351579725742
epoch:  18, loss: 0.003669709898531437
epoch:  19, loss: 0.003473159158602357
epoch:  20, loss: 0.003370204009115696
epoch:  21, loss: 0.0032094880007207394
epoch:  22, loss: 0.003102983348071575
epoch:  23, loss: 0.0030185815412551165
epoch:  24, loss: 0.0029386463575065136
epoch:  25, loss: 0.0028685154393315315
epoch:  26, loss: 0.002805098658427596
epoch:  27, loss: 0.0027765720151364803
epoch:  28, loss: 0.002698766998

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9593072951073677
Test metrics:  R2 = 0.6077649875745792


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="lsqrs")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.46425971388816833
epoch:  1, loss: 0.401960551738739
epoch:  2, loss: 0.09206625074148178
epoch:  3, loss: 0.038448747247457504
epoch:  4, loss: 0.029434405267238617
epoch:  5, loss: 0.02607242576777935
epoch:  6, loss: 0.019540250301361084
epoch:  7, loss: 0.017021453008055687
epoch:  8, loss: 0.016018956899642944
epoch:  9, loss: 0.015112737193703651
epoch:  10, loss: 0.014616473577916622
epoch:  11, loss: 0.014151406474411488
epoch:  12, loss: 0.013650784268975258
epoch:  13, loss: 0.01352870836853981
epoch:  14, loss: 0.013268041424453259
epoch:  15, loss: 0.013154316693544388
epoch:  16, loss: 0.013034339994192123
epoch:  17, loss: 0.01298060268163681
epoch:  18, loss: 0.012927505187690258
epoch:  19, loss: 0.012862861156463623
epoch:  20, loss: 0.012835001572966576
epoch:  21, loss: 0.012757360935211182
epoch:  22, loss: 0.012722483836114407
epoch:  23, loss: 0.012699829414486885
epoch:  24, loss: 0.012688647955656052
epoch:  25, loss: 0.012643719092011452
epoc

In [12]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6452744478849445
Test metrics:  R2 = 0.5148280929889342


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="cg")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.17065422236919403
epoch:  1, loss: 0.03961564227938652
epoch:  2, loss: 0.039380572736263275
epoch:  3, loss: 0.03914259374141693
epoch:  4, loss: 0.038950610905885696
epoch:  5, loss: 0.038705289363861084
epoch:  6, loss: 0.038686539977788925
epoch:  7, loss: 0.038686539977788925
epoch:  8, loss: 0.038686539977788925
epoch:  9, loss: 0.038686539977788925
epoch:  10, loss: 0.038686539977788925
epoch:  11, loss: 0.038686539977788925
epoch:  12, loss: 0.038686539977788925
epoch:  13, loss: 0.03868523985147476
epoch:  14, loss: 0.03868523985147476
epoch:  15, loss: 0.03868241235613823
epoch:  16, loss: 0.03868241235613823
epoch:  17, loss: 0.03868241235613823
epoch:  18, loss: 0.03868241235613823
epoch:  19, loss: 0.03868241235613823
epoch:  20, loss: 0.03868241235613823
epoch:  21, loss: 0.03868241235613823
epoch:  22, loss: 0.03868241235613823
epoch:  23, loss: 0.03868241235613823
epoch:  24, loss: 0.03868241235613823
epoch:  25, loss: 0.03868241235613823
epoch:  26, 

In [14]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -785.1995529216136
Test metrics:  R2 = -766.0307099565478


In [15]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="cg-trunc")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.5572741627693176
epoch:  1, loss: 0.042240649461746216
epoch:  2, loss: 0.038985274732112885
epoch:  3, loss: 0.03894118219614029
epoch:  4, loss: 0.03889721632003784
epoch:  5, loss: 0.03885005787014961
epoch:  6, loss: 0.03880290687084198
epoch:  7, loss: 0.03875789791345596
epoch:  8, loss: 0.038712695240974426
epoch:  9, loss: 0.03866713121533394
epoch:  10, loss: 0.038620270788669586
epoch:  11, loss: 0.03857389837503433
epoch:  12, loss: 0.03852605074644089
epoch:  13, loss: 0.03847617283463478
epoch:  14, loss: 0.03842432424426079
epoch:  15, loss: 0.03837199881672859
epoch:  16, loss: 0.03831802308559418
epoch:  17, loss: 0.038263533264398575
epoch:  18, loss: 0.03820684924721718
epoch:  19, loss: 0.03814903646707535
epoch:  20, loss: 0.03808898478746414
epoch:  21, loss: 0.038027502596378326
epoch:  22, loss: 0.03796346113085747
epoch:  23, loss: 0.037897463887929916
epoch:  24, loss: 0.03782827407121658
epoch:  25, loss: 0.037757523357868195
epoch:  26, los

In [16]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7441364102827406
Test metrics:  R2 = 0.6337153816304169


In [17]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="cr")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.17776714265346527
epoch:  1, loss: 0.039794016629457474
epoch:  2, loss: 0.03952260687947273
epoch:  3, loss: 0.039507906883955
epoch:  4, loss: 0.039507906883955
epoch:  5, loss: 0.039507906883955
epoch:  6, loss: 0.039507906883955
epoch:  7, loss: 0.039507906883955
epoch:  8, loss: 0.039507906883955
epoch:  9, loss: 0.039507906883955
epoch:  10, loss: 0.039507906883955
epoch:  11, loss: 0.039507906883955
epoch:  12, loss: 0.039507906883955
epoch:  13, loss: 0.039507906883955
epoch:  14, loss: 0.039507906883955
epoch:  15, loss: 0.039507906883955
epoch:  16, loss: 0.039507906883955
epoch:  17, loss: 0.039507906883955
epoch:  18, loss: 0.039507906883955
epoch:  19, loss: 0.039507906883955
epoch:  20, loss: 0.039507906883955
epoch:  21, loss: 0.039507906883955
epoch:  22, loss: 0.039507906883955
epoch:  23, loss: 0.039507906883955
epoch:  24, loss: 0.039507906883955
epoch:  25, loss: 0.039507906883955
epoch:  26, loss: 0.039507906883955
epoch:  27, loss: 0.03950790688

In [18]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -2074.8248199885907
Test metrics:  R2 = -1816.567439250327
